# Končni nabor podatkov
2020 - 2025


In [47]:
import pandas as pd
import unicodedata

In [48]:
df_air_quality = pd.read_csv('podatki_kakovost_zraka.csv', encoding='utf-8')
df_lung_cancer = pd.read_csv('podatki_pljucni_rak.csv', encoding='utf-8')

In [49]:
print(f"\nAir quality data shape: {df_air_quality.shape}")
print(f"Lung cancer data shape: {df_lung_cancer.shape}")


Air quality data shape: (1284, 8)
Lung cancer data shape: (1272, 4)


In [50]:
def normalize_text(text):
    """Normalize Unicode text to NFC (composed) form and strip whitespace"""
    if isinstance(text, str):
        # NFC = Canonical composition (precomposed characters like š, č, ž)
        return unicodedata.normalize('NFC', text).strip()
    return text

In [51]:
df_air_quality['Občina'] = df_air_quality['Občina'].apply(normalize_text)
df_lung_cancer['Občina'] = df_lung_cancer['Občina'].apply(normalize_text)

In [52]:
def standardize_municipality_name(name):
    """Standardize bilingual municipality names to use / separator"""
    replacements = {
        'Koper Capodistria': 'Koper/Capodistria',
        'IzolaIsola': 'Izola/Isola',
        'PiranPirano': 'Piran/Pirano',
        'Lendava_Lendva': 'Lendava/Lendva',
        'DobrovnikDobronak': 'Dobrovnik/Dobronak',
        'Hodoš Hodos': 'Hodoš/Hodos',
        'Sveti Andraž v Slov.goricah': 'Sveti Andraž v Slov. goricah',
    }
    return replacements.get(name, name)

In [53]:
df_air_quality['Občina'] = df_air_quality['Občina'].apply(standardize_municipality_name)
df_lung_cancer['Občina'] = df_lung_cancer['Občina'].apply(standardize_municipality_name)

In [54]:
air_municipalities = set(df_air_quality['Občina'].unique())
lung_municipalities = set(df_lung_cancer['Občina'].unique())

In [55]:
matching = air_municipalities & lung_municipalities
only_in_air = air_municipalities - lung_municipalities
only_in_lung = lung_municipalities - air_municipalities

print(f"\nMunicipalities in both datasets: {len(matching)}")
print(f"Municipalities ONLY in air quality: {len(only_in_air)}")
print(f"Municipalities ONLY in lung cancer: {len(only_in_lung)}")


Municipalities in both datasets: 212
Municipalities ONLY in air quality: 1
Municipalities ONLY in lung cancer: 0


In [56]:
df_merged = pd.merge(
    df_air_quality,
    df_lung_cancer,
    on=['Občina', 'Leto'],
    how='outer',  # Keep all records from both datasets
    indicator=True
)

print(f"\nMerged data shape: {df_merged.shape}")
print("\n" + "-" * 70)
print("MERGE STATISTICS")
print("-" * 70)
merge_stats = df_merged['_merge'].value_counts()
print(merge_stats)
print("\nExplanation:")
print(f"  • both       : {merge_stats.get('both', 0):,} records in both datasets (successfully matched)")
print(f"  • left_only  : {merge_stats.get('left_only', 0):,} records only in air quality data")
print(f"  • right_only : {merge_stats.get('right_only', 0):,} records only in lung cancer data")


Merged data shape: (1284, 11)

----------------------------------------------------------------------
MERGE STATISTICS
----------------------------------------------------------------------
_merge
both          1278
left_only        6
right_only       0
Name: count, dtype: int64

Explanation:
  • both       : 1,278 records in both datasets (successfully matched)
  • left_only  : 6 records only in air quality data
  • right_only : 0 records only in lung cancer data


In [57]:
nan_summary = df_merged.isnull().sum()
nan_cols = nan_summary[nan_summary > 0]

In [58]:
if len(nan_cols) > 0:
    print("\nColumns with NaN values:")
    for col, count in nan_cols.items():
        pct = (count / len(df_merged)) * 100
        print(f"  • {col:50s}: {count:5,} ({pct:5.1f}%)")

    print(f"\nTotal rows with at least one NaN: {df_merged.isnull().any(axis=1).sum():,} / {len(df_merged):,}")

    # Show examples
    rows_with_nan = df_merged[df_merged.isnull().any(axis=1)]
    print("\nExamples of rows with NaN (showing merge indicator):")
    print(rows_with_nan[['Občina', 'Leto', 'PM10 (µg/m³)', 'Novi primeri pljučnega raka', '_merge']].head(10).to_string())
else:
    print("\n✓ No NaN values found! Perfect merge!")


Columns with NaN values:
  • Novi primeri pljučnega raka                       :     6 (  0.5%)
  • Umrljivost zaradi pljučnega raka (0–74 let)       :     6 (  0.5%)

Total rows with at least one NaN: 6 / 1,284

Examples of rows with NaN (showing merge indicator):
                            Občina  Leto  PM10 (µg/m³) Novi primeri pljučnega raka     _merge
948  Sveta Trojica v Slov. Goricah  2020        16.087                         NaN  left_only
949  Sveta Trojica v Slov. Goricah  2021        18.176                         NaN  left_only
950  Sveta Trojica v Slov. Goricah  2022        16.866                         NaN  left_only
951  Sveta Trojica v Slov. Goricah  2023        15.850                         NaN  left_only
952  Sveta Trojica v Slov. Goricah  2024        15.267                         NaN  left_only
953  Sveta Trojica v Slov. Goricah  2025        12.463                         NaN  left_only


In [59]:
df_merged = df_merged.drop('_merge', axis=1)

In [60]:
output_file = 'koncni_nabor_podatkov.csv'
df_merged.to_csv(output_file, index=False, encoding='utf-8')

print("\n" + "=" * 70)
print("FINAL RESULTS")
print("=" * 70)
print(f"\n✓ Merged data saved to: koncni_nabor_podatkov.csv")
print(f"\nFinal dataset:")
print(f"  • Total rows: {len(df_merged):,}")
print(f"  • Unique municipalities: {df_merged['Občina'].nunique()}")
print(f"  • Years covered: {sorted(df_merged['Leto'].unique())}")
print(f"\nFirst 5 rows:")
print(df_merged.head().to_string())

print("\n" + "=" * 70)


FINAL RESULTS

✓ Merged data saved to: koncni_nabor_podatkov.csv

Final dataset:
  • Total rows: 1,284
  • Unique municipalities: 213
  • Years covered: [np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

First 5 rows:
       Občina  Leto  PM10 (µg/m³)  PM2.5 (µg/m³)  Ozon (µg/m³)  Žveplov dioksid (µg/m³)  Dušikov dioksid (µg/m³)  Ogljikov monoksid (µg/m³) Novi primeri pljučnega raka Umrljivost zaradi pljučnega raka (0–74 let)
0  Ajdovščina  2020        13.123          9.962        65.741                    0.953                    7.049                    166.459           65.15886105831194                                        32.4
1  Ajdovščina  2021        14.352          9.686        62.748                    0.968                    9.010                    202.976                       70.68                                       33.72
2  Ajdovščina  2022        14.033         10.994        66.272                    1.213           

In [61]:
df_merged

,Občina,Leto,PM10 (µg/m³),PM2.5 (µg/m³),Ozon (µg/m³),Žveplov dioksid (µg/m³),Dušikov dioksid (µg/m³),Ogljikov monoksid (µg/m³),Novi primeri pljučnega raka,Umrljivost zaradi pljučnega raka (0–74 let)
0,Ajdovščina,2020,13.123,9.962,65.741,0.953,7.049,166.459,65.15886105831194,32.4
1,Ajdovščina,2021,14.352,9.686,62.748,0.968,9.010,202.976,70.68,33.72
2,Ajdovščina,2022,14.033,10.994,66.272,1.213,4.981,150.996,69.42,28.04
3,Ajdovščina,2023,14.080,11.143,72.586,1.491,6.043,185.261,56.632,33.146
4,Ajdovščina,2024,12.143,9.746,68.555,0.448,3.254,169.675,52.625,31.166
...,...,...,...,...,...,...,...,...,...,...
1279,Žužemberk,2021,15.216,10.682,53.799,1.053,8.327,213.972,21.42,34.72
1280,Žužemberk,2022,14.213,11.441,64.656,1.150,5.685,152.865,42.94,39.73
1281,Žužemberk,2023,14.811,11.601,59.005,1.170,6.691,184.765,47.695,42.092
1282,Žužemberk,2024,12.245,9.845,61.795,0.318,4.202,172.294,66.861,45.057
